# Weather Prediction Model — Analysis & Validation
Gurgaon, India | Open-Meteo Archive API

**What this notebook does:**
1. Downloads historical weather data (2020–present)
2. Cleans and explores the data
3. Engineers features and shows correlation heatmap
4. Trains the same model used in production
5. Validates whether the model is actually working

In [ ]:
# ── Install dependencies ──────────────────────────────────
!pip install -q requests pandas numpy matplotlib seaborn scikit-learn xgboost

In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import date, timedelta
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 5)
print('Libraries loaded.')

## 1. Download Data from Open-Meteo

In [ ]:
# Downloads year-by-year — same logic as data/bootstrap.py
import time
HISTORY_START = date(2020, 1, 1)
HISTORY_END   = date.today() - timedelta(days=1)

BASE_URL = (
    'https://archive-api.open-meteo.com/v1/archive'
    '?latitude=28.4595&longitude=77.0266'
    '&hourly=temperature_2m,apparent_temperature,relative_humidity_2m,'
    'dew_point_2m,pressure_msl,cloudcover,windspeed_10m,'
    'winddirection_10m,windgusts_10m,precipitation,rain,weathercode'
)

COLUMN_MAP = {
    'time': 'recorded_at', 'temperature_2m': 'temperature',
    'apparent_temperature': 'feels_like', 'relative_humidity_2m': 'humidity',
    'dew_point_2m': 'dew_point', 'pressure_msl': 'pressure',
    'cloudcover': 'cloudcover', 'windspeed_10m': 'wind_speed',
    'winddirection_10m': 'wind_direction', 'windgusts_10m': 'wind_gusts',
    'precipitation': 'precipitation', 'rain': 'rain', 'weathercode': 'weather_main',
}

def download_chunk(start, end):
    import time
    url = f'{BASE_URL}&start_date={start}&end_date={end}'
    for attempt in range(5):
        resp = requests.get(url, timeout=60)
        if resp.status_code == 429:
            wait = 60 * (attempt + 1)
            print(f'\n  Rate limited. Waiting {wait}s (attempt {attempt+1}/5)...', flush=True)
            time.sleep(wait)
            continue
        resp.raise_for_status()
        break
    else:
        raise RuntimeError(f'Failed after 5 attempts (rate limited): {start} → {end}')
    df = pd.DataFrame(resp.json()['hourly'])
    df = df[list(COLUMN_MAP.keys())].rename(columns=COLUMN_MAP)
    df['recorded_at'] = pd.to_datetime(df['recorded_at'])
    num_cols = [c for c in df.columns if c != 'recorded_at']
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    return df

chunks = []
year = HISTORY_START.year
while date(year, 1, 1) <= HISTORY_END:
    start = date(year, 1, 1)
    end   = min(date(year, 12, 31), HISTORY_END)
    print(f'Downloading {start} → {end} ...', end=' ', flush=True)
    try:
        chunk = download_chunk(start, end)
        chunks.append(chunk)
        print(f'{len(chunk)} rows')
    except Exception as e:
        print(f'FAILED: {e}')
    time.sleep(3)  # be polite between chunks
    year += 1

df_raw = pd.concat(chunks, ignore_index=True)
df_raw = df_raw.sort_values('recorded_at').drop_duplicates('recorded_at').reset_index(drop=True)
print(f'\nTotal rows: {len(df_raw):,}  |  {df_raw["recorded_at"].min().date()} → {df_raw["recorded_at"].max().date()}')

## 2. Data Cleaning & Quality Check

In [ ]:
print('=== Shape ==='); print(df_raw.shape)
print('\n=== Null counts ===')
print(df_raw.isnull().sum())
print('\n=== Basic stats ===')
df_raw.describe().round(2)

In [ ]:
# Missing value heatmap — shows which columns/time periods have gaps
plt.figure(figsize=(14, 4))
missing = df_raw.set_index('recorded_at').isnull().resample('ME').mean() * 100
sns.heatmap(missing.T, cmap='Reds', linewidths=0.3, annot=False,
            cbar_kws={'label': '% missing'})
plt.title('Missing Data % by Month and Column')
plt.tight_layout(); plt.show()

In [ ]:
# Drop rows missing critical columns
df = df_raw.dropna(subset=['temperature', 'humidity', 'pressure']).copy()
# Fill minor gaps with forward fill
df = df.ffill()
print(f'After cleaning: {len(df):,} rows  (dropped {len(df_raw)-len(df):,})')

## 3. Exploratory Visualizations

In [ ]:
# Temperature over time
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

daily = df.set_index('recorded_at')['temperature'].resample('D').agg(['mean','min','max'])
axes[0].fill_between(daily.index, daily['min'], daily['max'], alpha=0.3, label='Daily range')
axes[0].plot(daily.index, daily['mean'], lw=1, label='Daily mean')
axes[0].set_title('Temperature Over Time — Gurgaon')
axes[0].set_ylabel('°C'); axes[0].legend()

# Monthly average
monthly = df.set_index('recorded_at')['temperature'].resample('ME').mean()
axes[1].bar(monthly.index, monthly.values, width=20, color=sns.color_palette('coolwarm', len(monthly)))
axes[1].set_title('Monthly Average Temperature')
axes[1].set_ylabel('°C')
plt.tight_layout(); plt.show()

In [ ]:
# Hourly temperature pattern — does the model have a real signal to learn?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hourly_avg = df.groupby(df['recorded_at'].dt.hour)['temperature'].mean()
axes[0].plot(hourly_avg.index, hourly_avg.values, marker='o')
axes[0].set_title('Average Temperature by Hour of Day')
axes[0].set_xlabel('Hour (UTC)'); axes[0].set_ylabel('°C')
axes[0].set_xticks(range(0, 24, 2))

monthly_avg = df.groupby(df['recorded_at'].dt.month)['temperature'].mean()
axes[1].bar(monthly_avg.index, monthly_avg.values,
            color=sns.color_palette('coolwarm', 12))
axes[1].set_title('Average Temperature by Month')
axes[1].set_xlabel('Month'); axes[1].set_ylabel('°C')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout(); plt.show()

## 4. Feature Engineering

In [ ]:
# Same feature engineering as ml/preprocess.py
def engineer_features(df):
    df = df.copy()
    dt = df['recorded_at']
    df['hour']        = dt.dt.hour
    df['day_of_week'] = dt.dt.dayofweek
    df['month']       = dt.dt.month
    df['season']      = (dt.dt.month % 12 // 3)
    df['is_daytime']  = dt.dt.hour.between(6, 18).astype(int)
    df['humidity_pressure_ratio'] = df['humidity'] / df['pressure'].replace(0, np.nan)
    date_col = dt.dt.date
    df['daily_temp_max'] = df.groupby(date_col)['temperature'].cummax()
    df['daily_temp_min'] = df.groupby(date_col)['temperature'].cummin()
    df['temp_lag_1h']  = df['temperature'].shift(1)
    df['temp_lag_3h']  = df['temperature'].shift(3)
    df['temp_lag_24h'] = df['temperature'].shift(24)
    df['temp_rolling_mean_6h'] = df['temperature'].rolling(6, min_periods=1).mean()
    df['temp_rolling_std_6h']  = df['temperature'].rolling(6, min_periods=1).std().fillna(0)
    return df

FEATURE_COLS = [
    'hour', 'day_of_week', 'month', 'season', 'is_daytime',
    'humidity', 'dew_point', 'pressure', 'cloudcover',
    'wind_speed', 'wind_direction', 'wind_gusts',
    'feels_like', 'precipitation', 'rain', 'weather_main',
    'humidity_pressure_ratio', 'daily_temp_max', 'daily_temp_min',
    'temp_lag_1h', 'temp_lag_3h', 'temp_lag_24h',
    'temp_rolling_mean_6h', 'temp_rolling_std_6h',
]

df_feat = engineer_features(df)
df_feat = df_feat.dropna(subset=FEATURE_COLS).reset_index(drop=True)
X = df_feat[FEATURE_COLS].values.astype('float32')
y = df_feat['temperature'].values.astype('float32')
print(f'Feature matrix: {X.shape}  |  Target: {y.shape}')

## 5. Correlation Heatmap

In [ ]:
corr_df = df_feat[FEATURE_COLS + ['temperature']].corr()

plt.figure(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_df, dtype=bool))
sns.heatmap(corr_df, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.3, annot_kws={'size': 7})
plt.title('Feature Correlation Heatmap', fontsize=14)
plt.tight_layout(); plt.show()

# Top correlations with target
print('\nTop correlations with temperature:')
print(corr_df['temperature'].drop('temperature').abs().sort_values(ascending=False).to_string())

## 6. Train Models & Compare (same as production)

In [ ]:
# Chronological split — never shuffle time-series
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
dates_test = df_feat['recorded_at'].iloc[split:].reset_index(drop=True)

print(f'Train: {len(X_train):,} rows  |  Test: {len(X_test):,} rows')
print(f'Test period: {dates_test.iloc[0].date()} → {dates_test.iloc[-1].date()}')

In [ ]:
models = {
    'GradientBoosting': GradientBoostingRegressor(n_estimators=200, max_depth=4, subsample=0.8, random_state=42),
    'RandomForest':     RandomForestRegressor(n_estimators=100, max_depth=12, n_jobs=-1, random_state=42),
    'Ridge':            Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
}

results = {}
predictions = {}

for name, model in models.items():
    print(f'Training {name}...', end=' ', flush=True)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results[name] = {'MAE': mae, 'RMSE': rmse}
    predictions[name] = preds
    print(f'MAE={mae:.4f}°C  RMSE={rmse:.4f}°C')

print('\n=== Summary ===')
pd.DataFrame(results).T.sort_values('MAE').round(4)

## 7. Is the Model Actually Working?

In [ ]:
# Pick best model for detailed analysis
best_name = min(results, key=lambda k: results[k]['MAE'])
best_preds = predictions[best_name]
print(f'Best model: {best_name}  MAE={results[best_name]["MAE"]:.4f}°C')

# Predicted vs Actual — last 7 days of test set
n = 24 * 7
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(dates_test[-n:], y_test[-n:],   label='Actual',    lw=1.5, alpha=0.9)
axes[0].plot(dates_test[-n:], best_preds[-n:], label='Predicted', lw=1.5, alpha=0.8, linestyle='--')
axes[0].set_title(f'{best_name} — Predicted vs Actual (last 7 days of test set)')
axes[0].set_ylabel('Temperature (°C)'); axes[0].legend()

# Residuals
residuals = best_preds - y_test
axes[1].plot(dates_test[-n:], residuals[-n:], color='tomato', lw=1, alpha=0.8)
axes[1].axhline(0, color='black', lw=0.8, linestyle='--')
axes[1].set_title('Residuals (Predicted − Actual)')
axes[1].set_ylabel('Error (°C)')
plt.tight_layout(); plt.show()

In [ ]:
# Error distribution — should be centred at 0
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=80, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(0, color='red', lw=1.5, linestyle='--')
axes[0].axvline(residuals.mean(), color='orange', lw=1.5, linestyle='--', label=f'Mean={residuals.mean():.3f}')
axes[0].set_title('Residual Distribution')
axes[0].set_xlabel('Error (°C)'); axes[0].legend()

# Scatter: predicted vs actual — perfect model = diagonal line
sample = np.random.choice(len(y_test), size=min(3000, len(y_test)), replace=False)
axes[1].scatter(y_test[sample], best_preds[sample], alpha=0.3, s=8, color='steelblue')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[1].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[1].set_title('Predicted vs Actual (scatter)')
axes[1].set_xlabel('Actual (°C)'); axes[1].set_ylabel('Predicted (°C)')
axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Baseline comparison — is ML better than just guessing "same as last hour"?
naive_preds = y_test[:-1]          # predict t+1 = t (naive lag-1 baseline)
naive_mae   = mean_absolute_error(y_test[1:], naive_preds)
model_mae   = results[best_name]['MAE']

print('=== Baseline Comparison ===')
print(f'Naive (same as last hour) MAE : {naive_mae:.4f}°C')
print(f'{best_name} MAE              : {model_mae:.4f}°C')
improvement = (naive_mae - model_mae) / naive_mae * 100
print(f'Improvement over naive        : {improvement:.1f}%')

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(['Naive baseline\n(lag-1)', best_name], [naive_mae, model_mae],
               color=['#e07070', '#5b9bd5'], width=0.4)
for bar, val in zip(bars, [naive_mae, model_mae]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.4f}°C', ha='center', va='bottom', fontsize=11)
ax.set_ylabel('MAE (°C) — lower is better')
ax.set_title('Model vs Naive Baseline')
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance — what is the model actually using?
best_model = models[best_name]
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
elif hasattr(best_model, 'named_steps'):
    print('Ridge is a linear model — showing coefficients instead')
    importances = np.abs(best_model.named_steps['model'].coef_)

imp_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=True)

plt.figure(figsize=(9, 8))
plt.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
plt.title(f'Feature Importance — {best_name}')
plt.xlabel('Importance')
plt.tight_layout(); plt.show()

In [ ]:
# Error by hour of day — when does the model struggle most?
err_df = pd.DataFrame({
    'hour':  df_feat['recorded_at'].iloc[split:].dt.hour.values,
    'error': np.abs(best_preds - y_test)
})
hourly_err = err_df.groupby('hour')['error'].mean()

plt.figure(figsize=(10, 4))
plt.bar(hourly_err.index, hourly_err.values, color=sns.color_palette('coolwarm', 24))
plt.title('Mean Absolute Error by Hour of Day')
plt.xlabel('Hour (UTC)'); plt.ylabel('MAE (°C)')
plt.xticks(range(0, 24))
plt.tight_layout(); plt.show()
print(f'Worst hour: {hourly_err.idxmax()}:00 UTC  (MAE={hourly_err.max():.4f}°C)')
print(f'Best  hour: {hourly_err.idxmin()}:00 UTC  (MAE={hourly_err.min():.4f}°C)')

In [ ]:
# Error by month — does it degrade in summer/monsoon?
err_month_df = pd.DataFrame({
    'month': df_feat['recorded_at'].iloc[split:].dt.month.values,
    'error': np.abs(best_preds - y_test)
})
monthly_err = err_month_df.groupby('month')['error'].mean()

plt.figure(figsize=(10, 4))
plt.bar(monthly_err.index, monthly_err.values, color=sns.color_palette('coolwarm', 12))
plt.title('Mean Absolute Error by Month')
plt.xlabel('Month'); plt.ylabel('MAE (°C)')
plt.xticks(range(1, 13), ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
plt.tight_layout(); plt.show()

In [ ]:
# Final verdict
print('=' * 50)
print('VERDICT')
print('=' * 50)
print(f'Best model  : {best_name}')
print(f'MAE         : {model_mae:.4f}°C')
print(f'RMSE        : {results[best_name]["RMSE"]:.4f}°C')
print(f'Naive MAE   : {naive_mae:.4f}°C')
print(f'Improvement : {improvement:.1f}% over naive baseline')
print()
if improvement > 20:
    print('✅ Model is genuinely learning — significantly better than naive baseline.')
elif improvement > 5:
    print('⚠️  Model is learning but improvement is modest. Consider more features or data.')
else:
    print('❌ Model barely beats naive baseline. Check feature engineering or data quality.')